In [1]:
import re
import numpy as np
import pandas as pd
import requests, sys
from BECancerResistome import beagle2vep

ENSEMBL VEP configurations


In [2]:
server = "https://rest.ensembl.org"
ext = "/vep/human/hgvs"
headers = {"Content-Type": "application/json", "Accept": "application/json"}

## Import Beagle sgRNA annotation


In [3]:
variants = pd.read_csv(f"data/BeagleCoelho/EG_collab-guides.txt", sep="\t")
variants.shape

(24752, 24)

In [4]:
variants["editor"] = (
    variants["Edit Type"].apply(lambda v: "ABE" if v == "A-G" else "CBE").values
)

In [5]:
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,sgRNA Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,editor
3587,ENST00000263967.4,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,179219714,sense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CBE
10716,ENST00000307102.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000015.10,ENSG00000169032,MAP2K1,+,...,66485175,sense,66485179A>G,A_5,883A>G,Arg295Gly,Missense,NaN,NaN,ABE
8716,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,55201179,antisense,55201193T>C,A_6,2952T>C,Asp984Asp,Silent,NaN,NaN,ABE
21657,ENST00000646891.2,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000157764,BRAF,-,...,140801503,sense,140801518G>A,C_5,754C>T,Arg252Ter,Nonsense,poly(T):TTTT,NaN,CBE
7239,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,55171176,sense,55171180C>T,C_5,1886C>T,Thr629Ile,Missense,NaN,NaN,CBE
15401,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,226386338,sense,226386352G>A,C_6,808C>T,Gln270Ter,Nonsense,NaN,NaN,CBE
369,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,4095437,sense,"4095452G>A, 4095449G>A","C_5, C_8","985-3C>T, 985C>T","(NC), Pro329Ser","Intron, Missense",NaN,NaN,CBE
2455,ENST00000263967.4,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,179201346,sense,"179201350C>T, 179201352C>T","C_5, C_7","623C>T, 625C>T","Thr208Ile, Leu209Leu","Missense, Silent",NaN,NaN,CBE
567,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,4099223,antisense,"4099230C>T;4099229C>T, 4099226C>T","C_8;C_7, C_4","890G>A;891G>A, 894G>A","Arg297Gln, Pro298Pro","Missense, Silent",NaN,NaN,CBE
2242,ENST00000263967.4,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,179199184,sense,"179199187A>G, 179199188A>G, 179199190A>G","A_4, A_5, A_7","352+10A>G, 352+11A>G, 352+13A>G","(NC), (NC), (NC)","Intron, Intron, Intron",NaN,NaN,ABE


## Format for ENSEMBL VEP input

Use the Beagle columns [input, Nucleotide Edits] and format as HGVS identifiers for ENSEMBL VEP. Examples include:

- `ENST00000618231.3:c.9G>C`
- `ENST00000471631.1:c.28_33delTCGCGG`
- `ENST00000285667.3:c.1047_1048insC`
- `5:g.140532G>C`


In [7]:
beagle2vep(variants.loc[7343])

['ENST00000275493.7:c.1940C>T',
 'ENST00000275493.7:c.1941C>T',
 'ENST00000275493.7:c.1943C>T']

In [8]:
variants["hgvs"] = [
    "-" if r["Nucleotide Edits"] is np.nan else beagle2vep(r)
    for _, r in variants.iterrows()
]
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,editor,hgvs
13146,ENST00000366794.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABE,-
24389,ENST00000649815.2,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000014.9,ENSG00000142208,AKT1,-,...,antisense,104776699C>T,C_6,247G>A,Val83Ile,Missense,NaN,NaN,CBE,[ENST00000649815.2:c.247G>A]
5589,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,antisense,"55152545G>A, 55152547G>A","C_6, C_4","629-1G>A, 630G>A","(NC), Leu210Leu","Splice-acceptor, Silent",NaN,NaN,CBE,"[ENST00000275493.7:c.629-1G>A, ENST00000275493..."
17068,ENST00000429687.8,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000014.9,ENSG00000129484,PARP2,+,...,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABE,-
12639,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,sense,"226361473G>A;226361472G>A, 226361470G>A;226361...","C_4;C_5, C_7;C_8","3032C>T;3033C>T, 3035C>T;3036C>T","Thr1011Ile, Ser1012Phe","Missense, Missense",NaN,NaN,CBE,"[ENST00000366794.10:c.3032C>T, ENST00000366794..."
19021,ENST00000621592.8,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000008.11,ENSG00000136997,MYC,+,...,antisense,"127738721G>A, 127738722G>A, 127738725G>A","C_8, C_7, C_4","504G>A, 505G>A, 508G>A","Gln168Gln, Ala169Thr, Ala170Thr","Silent, Missense, Missense",NaN,NaN,CBE,"[ENST00000621592.8:c.504G>A, ENST00000621592.8..."
4214,ENST00000263967.4,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,sense,179229314A>G,A_5,2538A>G,Gly846Gly,Silent,NaN,NaN,ABE,[ENST00000263967.4:c.2538A>G]
17846,ENST00000429687.8,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000014.9,ENSG00000129484,PARP2,+,...,sense,20356591A>G,A_6,1231A>G,Met411Val,Missense,NaN,NaN,ABE,[ENST00000429687.8:c.1231A>G]
960,ENST00000262948.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,antisense,4101114A>G;4101112A>G,A_6;A_4,610T>C;612T>C,Ser204Pro,Missense,NaN,NaN,ABE,"[ENST00000262948.10:c.610T>C, ENST00000262948...."
18389,ENST00000621592.8,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000008.11,ENSG00000136997,MYC,+,...,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CBE,-


In [9]:
variants_hgvs = [v for vs in variants["hgvs"] if vs != "-" for v in vs]

Write variants HGVS input to file


In [10]:
len(variants_hgvs)

30992

In [11]:
with open("data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt", "w") as f:
    for variant in variants_hgvs:
        f.write(f"{variant}\n")

In [12]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt

ENST00000262948.10:c.*12C>T
ENST00000262948.10:c.*14C>T
ENST00000262948.10:c.*7C>T
ENST00000262948.10:c.*8C>T
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.1203A>G
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.*1C>T
ENST00000262948.10:c.*9G>A
ENST00000262948.10:c.*10G>A


VEP command


In [13]:
# ./vep --af --af_1kg --af_gnomade --af_gnomadg --appris --biotype --buffer_size 500 --ccds --check_existing --distance 5000 --domains --gencode_primary --hgvs --mane --numbers --plugin OpenTargets,file=[path_to]/OTGenetics.tsv.gz --plugin CADD,snv=[path_to]/CADD_GRCh38_1.7_whole_genome_SNVs.tsv.gz,indels=[path_to]/CADD_GRCh38_1.7_InDels.tsv.gz --plugin REVEL,file=[path_to]/new_tabbed_revel_grch38.tsv.gz --plugin SpliceAI,snv=[path_to]/spliceai_scores.masked.snv.hg38.vcf.gz,indel=[path_to]/spliceai_scores.masked.indel.hg38.vcf.gz,snv_ensembl=[path_to]/spliceai_scores.raw.snv.ensembl_mane.grch38.110.vcf.gz --plugin Enformer,file=[path_to]/enformer_grch38.vcf.gz --plugin AlphaMissense,file=[path_to]/AlphaMissense_hg38.tsv.gz --plugin Blosum62 --plugin MaveDB,file=[path_to]/MaveDB_variants.tsv.gz --plugin ClinPred,file=[path_to]/ClinPred_hg38_sorted_tabbed.tsv.gz --plugin dbscSNV,[path_to]/dbscSNV1.1_GRCh38.txt.gz --plugin EVE,file=[path_to]/eve_merged.vcf.gz --plugin LOEUF,file=[path_to]/loeuf_dataset_grch38.tsv.gz,match_by=gene --plugin NMD --plugin AncestralAllele,[path_to]/homo_sapiens_ancestor_GRCh38_109.fa.gz --plugin MaxEntScan,[path_to]/maxentscan --plugin mutfunc,db=[path_to]/mutfunc_data.db,motif=1,int=1,mod=1,exp=1 --polyphen b --protein --pubmed --regulatory --show_ref_allele --sift b --species homo_sapiens --symbol --transcript_version --tsl --uniprot --uploaded_allele --var_synonyms --cache --input_file [input_data] --output_file [output_file]

In [14]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-VEP-HGVS-output.txt

#Uploaded_variation	Location	Allele	Consequence	IMPACT	SYMBOL	Gene	Feature_type	Feature	BIOTYPE	EXON	INTRON	HGVSc	HGVSp	cDNA_position	CDS_position	Protein_position	Amino_acids	Codons	Existing_variation	REF_ALLELE	UPLOADED_ALLELE	DISTANCE	STRAND	FLAGS	SYMBOL_SOURCE	HGNC_ID	MANE	MANE_SELECT	MANE_PLUS_CLINICAL	TSL	APPRIS	CCDS	ENSP	SWISSPROT	TREMBL	UNIPARC	UNIPROT_ISOFORM	SIFT	PolyPhen	DOMAINS	HGVS_OFFSET	AF	AFR_AF	AMR_AF	EAS_AF	EUR_AF	SAS_AF	gnomADe_AF	gnomADe_AFR_AF	gnomADe_AMR_AF	gnomADe_ASJ_AF	gnomADe_EAS_AF	gnomADe_FIN_AF	gnomADe_MID_AF	gnomADe_NFE_AF	gnomADe_REMAINING_AF	gnomADe_SAS_AF	gnomADg_AF	gnomADg_AFR_AF	gnomADg_AMI_AF	gnomADg_AMR_AF	gnomADg_ASJ_AF	gnomADg_EAS_AF	gnomADg_FIN_AF	gnomADg_MID_AF	gnomADg_NFE_AF	gnomADg_REMAINING_AF	gnomADg_SAS_AF	CLIN_SIG	SOMATIC	PHENO	PUBMED	VAR_SYNONYMS	MOTIF_NAME	MOTIF_POS	HIGH_INF_POS	MOTIF_SCORE_CHANGE	TRANSCRIPTION_FACTORS	OpenTargets_geneId	OpenTargets_l2g	CADD_PHRED	CADD_RAW	REVEL	SpliceAI_pred_DP_AG	SpliceAI_pred_DP_AL	SpliceAI_pred_DP_DG

Export HGVS annotated Beagle file


In [15]:
variants.to_csv(f"data/BeagleCoelho/EG_collab-guides-HGVS.txt", sep="\t", index=False)

In [16]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-HGVS.txt

Input	CRISPR Enzyme	Edit Type	Edit Window	Target Taxon	Target Assembly	Target Genome Sequence	Target Gene ID	Target Gene Symbol	Target Gene Strand	Target Transcript ID	Target Domain	sgRNA Sequence	sgRNA Context Sequence	PAM Sequence	sgRNA Sequence Start Pos. (global)	sgRNA Orientation	Nucleotide Edits (global)	Guide Edits	Nucleotide Edits	Amino Acid Edits	Mutation Category	Constraint Violations	Note	editor	hgvs
ENST00000262948.10	SpyoCas9NG	A-G	4..8	9606	GRCh38	NC_000019.10	ENSG00000126934	MAP2K2	-	ENST00000262948.10	CDS	CCGGGCTCCCTGCGTCCCGC	GTGGCCGGGCTCCCTGCGTCCCGCTGGTGA	TG	4090572	sense								ABE	-
ENST00000262948.10	SpyoCas9NG	C-T	4..8	9606	GRCh38	NC_000019.10	ENSG00000126934	MAP2K2	-	ENST00000262948.10	CDS	CCGGGCTCCCTGCGTCCCGC	GTGGCCGGGCTCCCTGCGTCCCGCTGGTGA	TG	4090572	sense	4090586G>A, 4090584G>A	C_6, C_8	*12C>T, *14C>T	(NC), (NC)	UTR, UTR			CBE	['ENST00000262948.10:c.*12C>T', 'ENST00000262948.10:c.*14C>T']
ENST00000262948.10	SpyoCas9NG	A-G	4..8	9606	GRCh38	NC_000019.10	ENSG000001

## [Optional] Rest API to query a single edit


In [17]:
vep_request = {"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}

In [18]:
# dict to string
vep_request = str(vep_request).replace("'", '"')
vep_request

'{"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}'

In [19]:
r = requests.post(
    server + ext,
    headers=headers,
    data=vep_request,
)

if not r.ok:
    r.raise_for_status()
    sys.exit()

decoded = r.json()
print(repr(decoded))

[{'colocated_variants': [{'strand': 1, 'id': 'rs1884202840', 'seq_region_name': '14', 'allele_string': 'A/G', 'end': 20357524, 'start': 20357524}], 'seq_region_name': '14', 'transcript_consequences': [{'strand': 1, 'biotype': 'protein_coding', 'impact': 'LOW', 'variant_allele': 'G', 'gene_symbol': 'PARP2', 'gene_id': 'ENSG00000129484', 'hgnc_id': 'HGNC:272', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'transcript_id': 'ENST00000250416', 'gene_symbol_source': 'HGNC'}, {'hgnc_id': 'HGNC:272', 'gene_id': 'ENSG00000129484', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'gene_symbol_source': 'HGNC', 'transcript_id': 'ENST00000429687', 'strand': 1, 'biotype': 'protein_coding', 'variant_allele': 'G', 'impact': 'LOW', 'gene_symbol': 'PARP2'}, {'strand': 1, 'distance': 1569, 'biotype': 'retained_intron', 'impact': 'MODIFIER', 'variant_allele': 'G', 'gene_symbol': 'PARP2', 'gene_id': 'ENSG00000129484', 'hgnc_id': 'HGNC:272', 'consequence_term

In [131]:
variant_vep = decoded[0]
variant_vep

{'strand': 1,
 'seq_region_name': '14',
 'end': 20357524,
 'transcript_consequences': [{'transcript_id': 'ENST00000250416',
   'gene_id': 'ENSG00000129484',
   'gene_symbol': 'PARP2',
   'variant_allele': 'G',
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'biotype': 'protein_coding',
   'strand': 1,
   'impact': 'LOW'},
  {'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'LOW',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'strand': 1,
   'biotype': 'protein_coding',
   'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000429687',
   'gene_symbol': 'PARP2'},
  {'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000527384',
   'gene_symbol': 'PARP2',
   'distance': 1569,
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'MODIFIER',
   'consequence_terms': ['dow